In [ ]:
from pystac_client import Client
from pystac import ItemCollection, Item
from odc.stac import stac_load
from pyproj import CRS
from xarray import Dataset, DataArray
import yaml
import geopandas as gpd
import logging
import os.path as op
import requests
from os import makedirs
import odc.stac
import planetary_computer
import matplotlib.pyplot as plt
import rioxarray as rio
from shapely.geometry import shape
import os

In [ ]:
# downloaded tiger line data for all counties in the conterminous us
url = "https://www2.census.gov/geo/tiger/TIGER2025/COUNTY/tl_2025_us_county.zip"
us_counties = gpd.read_file(url)


In [ ]:
montana_gdf = us_counties[us_counties['STATEFP'] == "30"].copy()
montana_gdf.head(), len(montana_gdf)

In [ ]:
montana_gdf.plot()

Alright, Let's take a few counties and then set up a comparative analysis of NBR 1 year apart 

In [ ]:
montana_gdf[montana_gdf['NAME'].str.contains('Ravalli')]

In [ ]:
rav_county = montana_gdf[montana_gdf['NAME'].str.contains('Ravalli')].copy()
# define crs
rav_county = rav_county.to_crs("EPSG: 4326")

poly = rav_county.dissolve()
poly = poly.to_crs("EPSG: 4326")


In [ ]:
# The whole county of Ravalli is a bit large for this exercise so we will
# use an example from St. Mary Peak

aoi = gpd.read_file(r"/home/bily/python/montana-wildfire/data/raw/stMary_aoi.shp")
aoi = aoi.to_crs("EPSG: 4326")

poly = aoi.dissolve()
poly = poly.to_crs("EPSG: 4326")

In [ ]:
def bbox_to_geom(bbox: list) -> dict:
    """Convert a bounding box to a geojson-formatted
    geometry.
    Arg:
        bbox (list): bouning box with coordinate order left, bottom, right, top
    Return:
        geometry (dict): geojson-formatted geometry, as a dict
    """
    geometry = {
      "type": "Polygon",
          "coordinates": [
            [
              [
                bbox[0],
                bbox[3]
              ],
              [
                bbox[0],
                bbox[1]
              ],
              [
                bbox[2],
                bbox[1]
              ],
              [
                bbox[2],
                bbox[3]
              ],
              [
                bbox[0],
                bbox[3]
              ]
            ]
          ],
  }

    return geometry

def make_datacube(items: ItemCollection, bands, resolution, bbox) -> Dataset:
    """Convert stac item collection into xarray Dataset object.
    Temporal compositing hard-coded to solar_day for now.
    Arg:
        items (pystac.ItemCollection): items to convert to datacube
    Return:
        dc (Dataset): space-time datacube
    """

    logging.info("Making datacube for items: %s", items.items)

    output_crs = CRS.from_epsg(items[0].properties["proj:code"].split(":")[1])

    dc = stac_load(
            items=items.items,
            bands=bands,
            resolution=resolution,
            groupby="solar_day",
            crs=output_crs,
            bbox=bbox,
            chunks={'x': 2048, 'y': 2048}
            )

    return dc

In [ ]:
bbox = aoi.total_bounds
cat = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", modifier=planetary_computer.sign_inplace)

Sometimes I define two date ranges a year apart and grab the average - to define an average value for a season and compare that to an average value against a different season (mainly looking at forest growth and harvest activities) - I kind of want to try searching and gathering a range and then seeing that change over time today

In [ ]:
# Define the search parameters for early summer 2024
start_date_old = "2024-06-01"
end_date_old = "2024-07-24"

# Define the search parameters for Fall 2024
start_date_young = "2024-09-01"
end_date_young = "2024-10-31"

# Define the Landsat collection ID
collection = "sentinel-2-l2a"

search_kwargs_old = {
    "max_items": 30,
    "limit": 10,
    "collections": collection,
    "datetime": f"{start_date_old}/{end_date_old}",
    "bbox": bbox,
    "query": {"eo:cloud_cover": {"lt": 10}}
}

search_kwargs_young = {
    "max_items": 30,
    "limit": 10,
    "collections": collection,
    "datetime": f"{start_date_young}/{end_date_young}",
    "bbox": bbox,
    "query": {"eo:cloud_cover": {"lt": 10}}
}

In [ ]:
# Perform the search for June 2015
search_old = cat.search(**search_kwargs_old)
# Perform the search for June 2016
search_young = cat.search(**search_kwargs_young)

# Get the search results
items_old = search_old.item_collection()
items_young = search_young.item_collection()


In [ ]:
items = items_old + items_young
if len(items_old) == 0 or len(items_young) == 0:
    print(f"Skipping {_} due to no items found")

In [ ]:
# check crs
epsg = items[0].properties["proj:code"]
epsg

In [ ]:
bands = ["B08", "B12", "SCL"]
chunks = {'x': 2048, 'y': 2048}

In [ ]:
# Filter items to only include those that fully intersect the polygon
filtered_items = []
for item in items:
    item_geom = shape(item.geometry)
    if item_geom.intersects(poly.geometry.iloc[0]):
        filtered_items.append(item)

print(f"Filtered {len(filtered_items)} Items that fully intersect the polygon:")

In [ ]:
dc = make_datacube(items=filtered_items, bands=bands, resolution=10, bbox=bbox)
dc.attrs["crs"] = f"epsg:{epsg}"

In [ ]:
import pandas as pd
# 1. Create a dictionary mapping DATE to OFFSET from your 133 items
# Use .date() to match the 'solar_day' grouping behavior
offset_dict = {}
for item in items:
    date_key = pd.to_datetime(item.datetime).date()
    baseline = item.properties.get("s2:processing_baseline", "02.00")
    val = -1000 if float(baseline) >= 4.0 else 0
    offset_dict[date_key] = val

# 2. Map those dictionary values to the 118 dates actually in your dc
# This aligns the sizes perfectly (118 -> 118)
dc_dates = pd.to_datetime(dc.time.values).date
aligned_offsets = [offset_dict[d] for d in dc_dates]

# 3. attach offset to datacube to offset in case of pre and post 2022 data update
dc.coords["boa_offset"] = ("time", aligned_offsets)

dc[bands[0]] = dc[bands[0]] + dc['boa_offset']
dc[bands[1]] = dc[bands[1]] + dc['boa_offset']

In [ ]:
nbr = (dc[bands[0]] - dc[bands[1]]) / (dc[bands[0]] + dc[bands[1]])

In [ ]:
fall_nbr = dc['nbr'].sel(time=slice(start_date_old, end_date_old)).mean(dim='time')
fall_nbr = fall_nbr.rio.clip(poly.geometry.values, poly.crs)
fall_nbr = fall_nbr.rio.write_crs(dc[bands[1]].rio.crs)

summer_nbr = dc['nbr'].sel(time=slice(start_date_young, end_date_young)).mean(dim='time')
summer_nbr = summer_nbr.rio.clip(poly.geometry.values, poly.crs)
summer_nbr = summer_nbr.rio.write_crs(dc[bands[1]].rio.crs)

In [ ]:
nbr_diff = summer_nbr - fall_nbr

In [ ]:
nbr_diff.compute()

In [ ]:
nbr_diff.plot()

Some things I would like to add would be

- a cloud mask to get rid of particularly cloudy scenes
- a smoothing moving window function